1. Load Silver Data
2. Dataset Overview
3. Column Audit
4. Null Value Analysis
5. Cardinality Analysis
6. Numeric Statistics
7. Categorical Statistics
8. Feature Decisions
9. Create ML-ready Dataset
10. Save Feature Table

# Data Audit & Feature Engineering

This notebook prepares the Silver dataset for Machine Learning.

Responsibilities:

- Perform feature audit
- Analyze null values
- Analyze feature distributions
- Identify useful features
- Create ML-ready dataset

Output:
feature_engineering_dataset

In [0]:
# important Imports
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import(
    StringIndexer,
    VectorAssembler,
    OneHotEncoder
)
from pyspark.ml.classification import(
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

In [0]:
## Reading data
from pyspark.sql import functions as F

silver_df = spark.table("`log-analytics`.silver.silver_logs")

print("="*60)
print("SILVER DATASET")
print("="*60)

print(f"Rows    : {silver_df.count():,}")
print(f"Columns : {len(silver_df.columns)}")

display(silver_df.limit(5))

In [0]:
## Dataset Schema
silver_df.printSchema()

In [0]:
display(silver_df.describe())

### Null Value Audit

In [0]:
null_summary = (
    silver_df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in silver_df.columns
    ])
)

display(null_summary)

### Null Percentage

In [0]:
total_rows = silver_df.count()

null_percent = []

for col in silver_df.columns:
    null_count = silver_df.filter(F.col(col).isNull()).count()

    null_percent.append((
        col,
        null_count,
        round((null_count/total_rows)*100, 2)
    ))

display(
    spark.createDataFrame(
        null_percent,
        ["Column", "Null Count", "Null %"]
    )
)

### Cardinality Analysis


In [0]:
cardinality = []

for col in silver_df.columns:
    cardinality.append((
        col,
        silver_df.select(col).distinct().count()
    ))

display(
    spark.createDataFrame(
        cardinality,
        ["Column", "Distinct Values"]
    )
)

### Numeric Statistics

In [0]:
numeric_cols = [
    "response_time",
    "status_code"
]

display(
    silver_df.select(numeric_cols).summary()
)

### Categorical Distribution

In [0]:
categorical_columns = [
    "service",
    "log_level",
    "http_method",
    "kafka_topic"
]

for col in categorical_columns:
    print("="*60)
    print(col.upper())
    display(
        silver_df.groupBy(col)
                 .count()
                 .orderBy(F.desc("count"))
    )

### Target Variable Distribution


In [0]:
display(
    silver_df.groupBy("is_anomaly").count()
)

# Feature Engineering for Machine Learning

This notebook transforms the Gold layer into an ML-ready feature dataset.

The Gold layer is designed for analytics, BI dashboards, and LLM applications, so it preserves all business-relevant information.

However, machine learning models require additional preprocessing before training. This notebook performs feature engineering by analyzing each feature and applying appropriate transformations.

## Feature Engineering Tasks

### 1. Drop Non-Predictive Features
Columns that contain unique identifiers or constant values are removed because they do not contribute to model learning.

### 2. Handle Missing Values
Missing values are handled according to the business meaning of each feature rather than applying a generic imputation strategy.

### 3. Feature Extraction
Timestamp features are decomposed into multiple time-based features such as hour, weekday, month, and weekend indicator.

### 4. Feature Creation
Additional informative features are generated, such as indicators for the presence of an instance ID.

### 5. Preserve LLM Features
Textual columns such as log messages are retained in the Gold layer for future LLM-based Root Cause Analysis, although they are excluded from classical machine learning models.

Output:
ml_feature_dataset

This is what we are doing.

Column	Action	Reason

event_id	Drop	Unique identifier

dataset_source	Drop	Constant value (OpenStack)

timestamp	Convert	Extract useful time features

event_timestamp	Drop after extraction	Raw timestamp no longer needed

event_date	Drop after extraction	Derived feature

request_id	Keep	Useful for LLM and traceability

user_id	Keep	Future versions may use it

project_id	Keep	Future versions may use it

instance_id	Keep + create indicator	Missingness itself is informative

client_ip	Keep	Useful for future feature engineering

http_path	Keep	Useful for API grouping later

message	Keep	Required for LLM

kafka metadata	Keep	Useful for monitoring and dashboards

response_category	Keep	Engineered feature

is_anomaly	Keep	Target

Notice that we are only dropping three columns initially:

event_id

dataset_source

raw timestamp columns (after extracting features)

Everything else remains.

In [0]:
## Create ML Dataset
silver_df = spark.table("`log-analytics`.silver.silver_logs")
ml_df = silver_df

In [0]:
## Create Time Features
from pyspark.sql import functions as F
ml_df = (
    ml_df
    .withColumn("event_hour", F.hour("event_timestamp"))
    .withColumn("event_day", F.hour("event_timestamp"))
    .withColumn("event_month", F.hour("event_timestamp"))
    .withColumn("event_weekday", F.hour("event_timestamp"))
    .withColumn(
        "is_weekend", 
        F.when(
            F.dayofweek("event_timestamp").isin([1,7]),
        1
        ).otherwise(0)
    )
)

In [0]:
## Instance Feature
ml_df = ml_df.withColumn(
    "instance_exists",
    F.when(
        F.col("instance_id").isNull(),
        0
    ).otherwise(1)
)

In [0]:
## Handle Nulls
ml_df = (
    ml_df 
    .fillna({
        "http_method":"NO_HTTP",
        "http_path":"NO_PATH",
        "status_code":-1,
        "response_time":-1
    })
)
display(ml_df)

In [0]:
## Drop Truly Unnecessary Columns
ml_df = ml_df.drop(
    "event_id",
    "dataset_source",
    "timestamp",
    "event_timestamp",
    "event_date"
)

In [0]:
## Validation
print("="*60)
print("ML FEATURE DATASET")
print("="*60)

print(f"Rows      : {ml_df.count():,}")
print(f"Columns   : {len(ml_df.columns)}")

display(ml_df.limit(10))

In [0]:
## Save ML Feature Dataset
ml_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("`log-analytics`.gold.feature_engineering_dataset")

print("✓ ML Feature Dataset saved to: log-analytics.gold.feature_engineering_dataset")